# Mersenne Prime Cryptosystem

## SECTION 1.2 OUR CRYPTOSYSTEM

## Sem error correction: só para ver como funciona

In [1]:
# SEM CÓDIGO DE CORREÇÃO DE ERROS

import secrets # Para geração de números aleatórios seguros

def low_hamming_weight_number(n, weight):
    """Gera um número de n bits com um Peso de Hamming específico."""
    bits = [0] * n
    indices = set()
    while len(indices) < weight:
        indices.add(secrets.randbelow(n)) #escolhe indices aleatorios para colocar o bit 1 - garantindo o peso de Hamming
    for i in indices:
        bits[i] = 1
    return int("".join(map(str, bits)), 2)

def mersenne_encrypt(m_bits, n, weight, p, R, T):
    """
    weight: número de bits 1 em A, B1, B2

    Simulação da criptografia:
    C1 = A*R + B1 (mod p)
    C2 = (A*T + B2) XOR E(m)
    """
    # Escolha de A, B1, B2 com baixo peso de Hamming
    A = low_hamming_weight_number(n, weight)
    B1 = low_hamming_weight_number(n, weight)
    B2 = low_hamming_weight_number(n, weight)
    
    # C1 = A * R + B1 mod p
    C1 = (A * R + B1) % p
    
    # C2 = (A * T + B2) XOR m #sem error correction 
    pad = (A * T + B2) % p
    C2 = pad ^ m_bits # XOR para combinar o pad com a mensagem
    
    return (C1, C2)

def mersenne_decrypt(C, F, p):
    """
    Desencriptar a mensagem:
    C2_star = C1 * F (mod p)
    m = D(C2 XOR C2_star)
    """
    C1, C2 = C
    C2_star = (C1 * F) % p
    
    # m_recuperado = C2 XOR C2_star
    m_bits = C2 ^ C2_star
    return m_bits

# --- Exemplo de Uso ---
n = 17 # Usando n pequeno para exemplo (2^17 - 1)
p = 2**n - 1

# 1. Geração de Chaves
F = low_hamming_weight_number(n, 3) # Secret Key
G = low_hamming_weight_number(n, 3)
R = secrets.randbits(n) # Random n-bit string
T = (F * R + G) % p    # Public Key: (R, T)

# 2. Mensagem (m)
m = int("10101010101010101", 2) 

# 3. Execução
C = mersenne_encrypt(m, n, 3, p, R, T)
m_final = mersenne_decrypt(C, F, p)

print(f"Mensagem Original:  {bin(m)}")
print(f"Mensagem Recuperada: {bin(m_final)}")

Mensagem Original:  0b10101010101010101
Mensagem Recuperada: 0b11111000111011110


## Com error correction, repetition code

In [2]:
import secrets

def low_hamming_weight_number(n, weight):
    bits = [0] * n
    indices = set()
    while len(indices) < weight:
        indices.add(secrets.randbelow(n))
    for i in indices:
        bits[i] = 1
    return int("".join(map(str, bits)), 2)

#codigo de reptição: subsitutui cada bit da mensagem por um bloco de bits (ex: 3 bits) para aumentar a redundância e permitir correção de erros
def repetition_encode(m_bits, k, n):
    """
    E(m): Repete cada bit da mensagem 'rep' vezes para preencher os n bits.
    """
    rep = n // k
    encoded = 0
    for i in range(k):
        # Isola o bit i da mensagem
        bit = (m_bits >> i) & 1
        if bit:
            # Cria um bloco de 'rep' bits 1 e coloca na posição i*rep
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded


def repetition_decode(c_star, k, n):
    """
    D(C): Divide os n bits em blocos e faz uma votação majoritária.
    """
    rep = n // k
    decoded = 0
    for i in range(k):
        # Extrai o bloco de bits correspondente ao bit original i
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        # Conta quantos bits 1 existem (Hamming Weight do bloco)
        count_ones = bin(block).count('1')
        # Se mais de metade forem 1, o bit original era 1
        if count_ones > (rep // 2):
            decoded |= (1 << i)
    return decoded


def mersenne_encrypt(m_original, k, n, weight, p, R, T):
    # 1. Aplicar o Código de Repetição (E)
    m_encoded = repetition_encode(m_original, k, n)
    
    A = low_hamming_weight_number(n, weight)
    B1 = low_hamming_weight_number(n, weight)
    B2 = low_hamming_weight_number(n, weight)
    
    C1 = (A * R + B1) % p
    
    # 2. XOR com a mensagem expandida
    pad = (A * T + B2) % p
    C2 = pad ^ m_encoded 
    
    return (C1, C2)

def mersenne_decrypt(C, F, k, n, p):
    C1, C2 = C
    C2_star = (C1 * F) % p
    
    # 1. XOR para obter a mensagem com ruído
    diff = C2 ^ C2_star
    
    # 2. Aplicar a Decodificação por votação (D)
    m_recuperada = repetition_decode(diff, k, n)
    return m_recuperada

# --- Exemplo de Uso ---
n = 127 # n maior para permitir mais repetições e melhor correção
p = 2**n - 1
k = 7   # Vamos enviar apenas 7 bits de informação real
weight = 4 # Peso de Hamming das chaves/ruído

# 1. Geração de Chaves
F = low_hamming_weight_number(n, weight)
G = low_hamming_weight_number(n, weight)
R = secrets.randbits(n)
T = (F * R + G) % p

# 2. Mensagem (m) de k bits (ex: 0b1010101)
m_original = 0b1010101 

# 3. Execução
C = mersenne_encrypt(m_original, k, n, weight, p, R, T)
m_final = mersenne_decrypt(C, F, k, n, p)

print(f"Mensagem Original (k={k}): {bin(m_original)}")
print(f"Mensagem Recuperada:      {bin(m_final)}")
print(f"Sucesso: {m_original == m_final}")

Mensagem Original (k=7): 0b1010101
Mensagem Recuperada:      0b1010100
Sucesso: False


## BIT BY BIT encryption

In [3]:
import secrets
import math

def low_hamming_weight_number(n, h):
    """Gera um número de n bits com peso de Hamming exatamente h."""
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def hamming_weight(x):
    """Calcula o Peso de Hamming (quantidade de bits '1')."""
    return bin(x).count('1')

def generate_keys(n, p):
    """
    Gera a Chave Pública (H) e a Chave Secreta (G).
    H = F * G^-1 (mod p)
    """
    # Calcula o limite de h baseado em n: 4h^2 < n <= 16h^2
    # h_min > sqrt(n/16) e h_max < sqrt(n/4)
    h_min = math.ceil(math.sqrt(n / 16))
    h_max = math.floor(math.sqrt(n / 4))
    
    # Escolha aleatória de h dentro do limite
    h = secrets.choice(range(h_min, h_max + 1))
    
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    
    # H = F / G mod p
    try:
        inv_G = pow(G, -1, p)
        H = (F * inv_G) % p
    except ValueError:
        # Caso raro onde G não é inversível, tentamos novamente
        return generate_keys(n, p)
        
    return (H, G, h)

def encrypt(pk_H, b, n, p, h):
    """
    C = (-1)^b * (A * H + B) mod p
    """
    A = low_hamming_weight_number(n, h)
    B = low_hamming_weight_number(n, h)
    
    val = (A * pk_H + B) % p
    
    if b == 0:
        return val
    else:
        # (-1) * val no mundo modular é p - val
        return (p - val) % p

def decrypt(C, sk_G, n, p, h):
    """
    Decriptação baseada na distância de Hamming.
    """
    # d = Ham(C * G mod p)
    target = (C * sk_G) % p
    d = hamming_weight(target)
    
    threshold = 2 * (h**2)
    
    if d <= threshold:
        return 0
    elif d >= n - threshold:
        return 1
    else:
        return None # Caso "?" (Indeterminado)

# --- Exemplo de Execução ---

# 1. Parâmetros (p deve ser um primo de Mersenne)
n = 127 
p = 2**n - 1

# 2. Setup de Chaves
pk, sk, h_usado = generate_keys(n, p)

# 3. Teste com bit 0
bit_original = 1
criptograma = encrypt(pk, bit_original, n, p, h_usado)
bit_recuperado = decrypt(criptograma, sk, n, p, h_usado)

print(f"Parâmetros: n={n}, h={h_usado}")
print(f"Bit Original: {bit_original}")
print(f"Criptograma (C): {hex(criptograma)[:20]}...")
print(f"Bit Recuperado: {bit_recuperado}")

Parâmetros: n=127, h=4
Bit Original: 1
Criptograma (C): 0x187a22e93abdda4333...
Bit Recuperado: 1


## Section 4: Semantically Secure Public-Key Cryptosystem com repetition code

In [5]:
import secrets
import math

def low_hamming_weight_number(n, h):
    """Gera uma string de n bits com peso de Hamming exatamente h."""
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def repetition_encode(m_bits, k, n):
    """E(m): Código de correção de erro simples (Repetição)."""
    rep = n // k
    encoded = 0
    for i in range(k):
        bit = (m_bits >> i) & 1
        if bit:
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded

def repetition_decode(c_star, k, n):
    """D(C*): Decodificação por votação majoritária."""
    rep = n // k
    decoded = 0
    for i in range(k):
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        if bin(block).count('1') > (rep // 2):
            decoded |= (1 << i)
    return decoded

def is_prime(n):
    """Small helper to keep the example self-contained."""
    if n < 2: return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0: return False
    return True

def get_mersenne_exponent(h):
    """
    Finds the smallest Mersenne exponent n such that n > 10 * h^2.
    Common Mersenne exponents: 31, 61, 127, 521, 607, 1279, 2203, 2281, 3217, 4253, 4423, 9689, 9941, 11213, 19937...
    """
    # A list of known Mersenne exponents
    mersenne_exponents = [31, 61, 127, 521, 607, 1279, 2203, 2281, 3217, 4253, 4423, 9689, 9941, 11213, 19937, 21701]
    
    n_min = 10 * (h**2)
    
    for n in mersenne_exponents:
        if n >= n_min:
            return n
            
    # Fallback if h is very large: this won't necessarily be a Mersenne Prime exponent 
    # but follows the logic of the search.
    return n_min 

def generate_keys(lambda_param):
    """
    Key Generation based on Mersenne Prime n.
    h = lambda
    n is a Mersenne exponent such that n > 10h^2
    """
    h = lambda_param
    
    # 1. Select a valid Mersenne exponent n
    n = get_mersenne_exponent(h)
    
    # 2. Define the Mersenne Prime p = 2^n - 1
    p = (1 << n) - 1
    
    # 3. Generate low Hamming weight elements
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    R = secrets.randbits(n)
    
    # 4. pk := (R, T) where T = (F * R + G) mod p
    # In Mersenne arithmetic, (x * y) % p can be optimized, 
    # but Python's % handles large integers automatically.
    T = (F * R + G) % p
    
    return (R, T), F, n, p, h

def encrypt(pk, m, n, p, h, k):
    """
    Enc(pk, m) := (C1, C2)
    C1 = A * R + B1
    C2 = (A * T + B2) XOR E(m)
    """
    R, T = pk
    A = low_hamming_weight_number(n, h)
    B1 = low_hamming_weight_number(n, h)
    B2 = low_hamming_weight_number(n, h)
    
    # E(m)
    encoded_m = repetition_encode(m, k, n)
    
    C1 = (A * R + B1) % p
    pad = (A * T + B2) % p
    C2 = pad ^ encoded_m
    
    return (C1, C2)

def decrypt(C, sk_F, n, p, h, k):
    """
    Dec(sk, C) := D((F * C1) XOR C2)
    """
    C1, C2 = C
    
    # Calcula o termo que deve cancelar o pad: F * C1
    # F * (A * R + B1) = A * F * R + F * B1
    f_c1 = (sk_F * C1) % p
    
    # Extrai a mensagem com ruído usando XOR
    noisy_m = f_c1 ^ C2
    
    # D(noisy_m)
    return repetition_decode(noisy_m, k, n)

# --- Execução ---
# Parâmetro de segurança (ex: lambda = 16)
lambda_sec = 16 
k_message_size = 16 # Tamanho do bloco m

# 1. Setup
pk, sk, n, p, h = generate_keys(lambda_sec)

# 2. Mensagem m (Ex: 16 bits aleatórios)
m_original = secrets.randbits(k_message_size)

# 3. Cifrar
C = encrypt(pk, m_original, n, p, h, k_message_size)

# 4. Decifrar
m_recuperada = decrypt(C, sk, n, p, h, k_message_size)

print(f"--- Sistema de Bloco (Mersenne) ---")
print(f"n: {n}, h: {h}, k: {k_message_size}")
print(f"Mensagem Original:  {bin(m_original)}")
print(f"Mensagem Recuperada: {bin(m_recuperada)}")
print(f"Sucesso: {m_original == m_recuperada}")

--- Sistema de Bloco (Mersenne) ---
n: 3217, h: 16, k: 16
Mensagem Original:  0b1111001110110
Mensagem Recuperada: 0b1111001110110
Sucesso: True


## Mersenne Key Encapsulation Mechanism

### Testar para ver se o mersenne key encapsulation mechanism funciona

In [ ]:
import secrets
import math
import hashlib

# --- Funções Auxiliares e Aritmética de Mersenne ---

def low_hamming_weight_number(n, h):
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def repetition_encode(m_bits, k, n):
    rep = n // k
    encoded = 0
    for i in range(k):
        bit = (m_bits >> i) & 1
        if bit:
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded

def repetition_decode(c_star, k, n):
    rep = n // k
    decoded = 0
    for i in range(k):
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        if bin(block).count('1') > (rep // 2):
            decoded |= (1 << i)
    return decoded

def get_mersenne_exponent(h):
    mersenne_exponents = [31, 61, 127, 521, 607, 1279, 2203, 2281, 3217, 4253, 4423, 9689, 9941, 11213, 19937]
    n_min = 10 * (h**2)
    for n in mersenne_exponents:
        if n >= n_min:
            return n
    return n_min 

# --- Oráculos Aleatórios (Expandable Hash) ---

def expandable_hash_to_hamming(seed_int, n, h, lambda_param, salt=b''):
    # Converte o inteiro K para bytes para o Hash
    seed_bytes = seed_int.to_bytes((lambda_param + 7) // 8, 'big')
    xo_function = hashlib.shake_256(seed_bytes + salt)
    
    indices = set()
    byte_counter = 0
    while len(indices) < h:
        bytes_needed = (n.bit_length() + 7) // 8
        raw_bytes = xo_function.digest(byte_counter + bytes_needed)[byte_counter:]
        idx = int.from_bytes(raw_bytes, 'big') % n
        indices.add(idx)
        byte_counter += bytes_needed
        
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def H1(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H1')
def H2(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H2')
def H3(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H3')

# --- Core: Geração, Encapsulamento e Decapsulamento ---

def generate_keys(lambda_param):
    h = lambda_param
    n = get_mersenne_exponent(h)
    p = (1 << n) - 1
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    R = secrets.randbits(n)
    T = (F * R + G) % p
    return (R, T), F, n, p, h

def key_encapsulation(pk, lambda_param, n, p, h):
    # 1. Pick a uniformly random lambda-bit string K
    K = secrets.randbits(lambda_param)

    # 2. Let A = H1(K), B1 = H2(K), and B2 = H3(K)
    A = H1(K, n, h, lambda_param)
    B1 = H2(K, n, h, lambda_param)
    B2 = H3(K, n, h, lambda_param)
    
    # 3. C1 = A * R + B1; C2 = E(K) XOR (A * T + B2)
    C1 = (A * pk[0] + B1) % p
    encoded_K = repetition_encode(K, lambda_param, n)
    # Importante: o XOR ocorre após o mod p do termo (A*T + B2)
    C2 = encoded_K ^ ((A * pk[1] + B2) % p)
    
    return (C1, C2), K

def key_decapsulation(C, sk_F, pk, lambda_param, n, p, h):
    # 1. K' = D((F * C1) XOR C2)
    # F * C1 mod p cancela os termos de erro
    dec_val = ((sk_F * C[0]) % p) ^ C[1]
    K_prime = repetition_decode(dec_val, lambda_param, n)

    # 2. Re-deriva os parâmetros para verificação (A', B1', B2')
    A_prime = H1(K_prime, n, h, lambda_param)
    B1_prime = H2(K_prime, n, h, lambda_param)
    B2_prime = H3(K_prime, n, h, lambda_param)

    # 3. Reconstrói C' para verificar integridade
    C1_check = (A_prime * pk[0] + B1_prime) % p
    encoded_K_prime = repetition_encode(K_prime, lambda_param, n)
    C2_check = encoded_K_prime ^ ((A_prime * pk[1] + B2_prime) % p)

    # 4. Se C' == C, a chave é válida (Segurança IND-CCA)
    if C1_check == C[0] and C2_check == C[1]:
        return K_prime
    return None

# --- Execução do Teste ---

# Configurações
lambda_sec = 16  # h = 16

# 1. Setup (Geração de Chaves)
pk, sk, n, p, h = generate_keys(lambda_sec)
print(f"--- Parâmetros --- \nn: {n}, h: {h}, p: Mersenne Prime")

# 2. Encapsulamento (Criação do Segredo)
(C1, C2), K_original = key_encapsulation(pk, lambda_sec, n, p, h)
print(f"\n--- Encapsulamento --- \nK original: {bin(K_original)}")

# 3. Decapsulamento (Recuperação do Segredo)
K_recuperado = key_decapsulation((C1, C2), sk, pk, lambda_sec, n, p, h)

# 4. Verificação final
if K_recuperado is not None:
    print(f"\n--- Sucesso! --- \nK recuperado: {bin(K_recuperado)}")
    print(f"As chaves coincidem? {K_original == K_recuperado}")
else:
    print("\n--- Erro! --- \nFalha na verificação de integridade.")

--- Parâmetros --- 
n: 3217, h: 16, p: Mersenne Prime

--- Encapsulamento --- 
K original: 0b11010001110001

--- Sucesso! --- 
K recuperado: 0b11010001110001
As chaves coincidem? True


### Cifrar a chave 

In [14]:
import secrets
import math
import hashlib

# --- [Funções de Base: Mersenne e Repetição] ---

def low_hamming_weight_number(n, h):
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def repetition_encode(m_bits, k, n):
    rep = n // k
    encoded = 0
    for i in range(k):
        bit = (m_bits >> i) & 1
        if bit:
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded

def repetition_decode(c_star, k, n):
    rep = n // k
    decoded = 0
    for i in range(k):
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        if bin(block).count('1') > (rep // 2):
            decoded |= (1 << i)
    return decoded

def get_mersenne_exponent(h):
    # n > 10 * h^2
    mersenne_exponents = [31, 61, 127, 521, 607, 1279, 2203, 2281, 3217, 4253, 4423, 9689, 9941, 11213, 19937]
    n_min = 10 * (h**2)
    for n in mersenne_exponents:
        if n >= n_min: return n
    return n_min

# --- [Oráculos Aleatórios H1, H2, H3] ---

def expandable_hash_to_hamming(seed_int, n, h, lambda_param, salt=b''):
    seed_bytes = seed_int.to_bytes((lambda_param + 7) // 8, 'big')
    xo_function = hashlib.shake_256(seed_bytes + salt)
    indices = set()
    byte_counter = 0
    while len(indices) < h:
        bytes_needed = (n.bit_length() + 7) // 8
        raw_bytes = xo_function.digest(byte_counter + bytes_needed)[byte_counter:]
        idx = int.from_bytes(raw_bytes, 'big') % n
        indices.add(idx)
        byte_counter += bytes_needed
    num = 0
    for i in indices: num |= (1 << i)
    return num

def H1(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H1')
def H2(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H2')
def H3(K, n, h, lp): return expandable_hash_to_hamming(K, n, h, lp, b'H3')

# --- [Core: Mersenne KEM] ---

def generate_keys(lambda_param):
    h = lambda_param
    n = get_mersenne_exponent(h)
    p = (1 << n) - 1
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    R = secrets.randbits(n)
    T = (F * R + G) % p
    return (R, T), F, n, p, h

def key_encapsulation(pk, lambda_param, n, p, h):
    K = secrets.randbits(lambda_param)
    A = H1(K, n, h, lambda_param)
    B1 = H2(K, n, h, lambda_param)
    B2 = H3(K, n, h, lambda_param)
    C1 = (A * pk[0] + B1) % p
    C2 = repetition_encode(K, lambda_param, n) ^ ((A * pk[1] + B2) % p)
    return (C1, C2), K

def key_decapsulation(C, sk_F, pk, lambda_param, n, p, h):
    val_to_decode = ((sk_F * C[0]) % p) ^ C[1]
    K_prime = repetition_decode(val_to_decode, lambda_param, n)
    A_p = H1(K_prime, n, h, lambda_param)
    B1_p = H2(K_prime, n, h, lambda_param)
    B2_p = H3(K_prime, n, h, lambda_param)
    C1_check = (A_p * pk[0] + B1_p) % p
    C2_check = repetition_encode(K_prime, lambda_param, n) ^ ((A_p * pk[1] + B2_p) % p)
    if C1_check == C[0] and C2_check == C[1]:
        return K_prime
    return None

# --- [Transformação em PKE Completo (Criptografia Híbrida)] ---

def encrypt_pke(pk, message_str, lambda_param, n, p, h):
    """Transforma o KEM em criptografia de chave pública (PKE)."""
    # 1. Encapsula uma chave K aleatória
    (C1, C2), K = key_encapsulation(pk, lambda_param, n, p, h)
    
    # 2. Deriva uma chave simétrica de K para cifrar a mensagem real
    K_bytes = K.to_bytes((lambda_param + 7) // 8, 'big')
    sym_key = hashlib.sha256(K_bytes).digest()
    
    # 3. Cifra a mensagem (Exemplo simples com XOR para demonstração)
    m_bytes = message_str.encode()
    # Criamos um fluxo de bits a partir da sym_key para o XOR
    stream = hashlib.shake_256(sym_key).digest(len(m_bytes))
    ciphertext_msg = bytes([b ^ s for b, s in zip(m_bytes, stream)])
    
    return (C1, C2), ciphertext_msg

def decrypt_pke(C_envelope, ciphertext_msg, sk_F, pk, lambda_param, n, p, h):
    """Desencapsula K e decifra a mensagem real."""
    # 1. Recupera K usando Decapsulation
    K_prime = key_decapsulation(C_envelope, sk_F, pk, lambda_param, n, p, h)
    if K_prime is None:
        raise ValueError("Falha na integridade: Chave ou envelope corrompido!")
    
    # 2. Deriva a mesma chave simétrica
    K_bytes = K_prime.to_bytes((lambda_param + 7) // 8, 'big')
    sym_key = hashlib.sha256(K_bytes).digest()
    
    # 3. Decifra a mensagem
    stream = hashlib.shake_256(sym_key).digest(len(ciphertext_msg))
    original_bytes = bytes([b ^ s for b, s in zip(ciphertext_msg, stream)])
    
    return original_bytes.decode()

# --- [Execução] ---

lambda_sec = 16
pk, sk, n, p, h = generate_keys(lambda_sec)

mensagem = "Esta é uma mensagem secreta usando Mersenne!"
print(f"Mensagem Original: {mensagem}")

# Enviar
envelope, msg_cifrada = encrypt_pke(pk, mensagem, lambda_sec, n, p, h)

# Receber
try:
    mensagem_final = decrypt_pke(envelope, msg_cifrada, sk, pk, lambda_sec, n, p, h)
    print(f"Mensagem Decifrada: {mensagem_final}")
except ValueError as e:
    print(e)

Mensagem Original: Esta é uma mensagem secreta usando Mersenne!
Mensagem Decifrada: Esta é uma mensagem secreta usando Mersenne!


### errors

In [19]:
def test_error_rate(num_tests, lambda_param, message_len=16):
    pk, sk, n, p, h = generate_keys(lambda_param)

    errors = 0

    for i in range(num_tests):
        # gerar mensagem aleatória
        message = ''.join(secrets.choice('01') for _ in range(message_len))

        try:
            envelope, msg_cifrada = encrypt_pke(pk, message, lambda_param, n, p, h)
            msg_final = decrypt_pke(envelope, msg_cifrada, sk, pk, lambda_param, n, p, h)

            if msg_final != message:
                errors += 1

        except Exception:
            # qualquer falha conta como erro
            errors += 1

    error_rate = errors / num_tests

    print(f"Testes realizados: {num_tests}")
    print(f"Erros: {errors}")
    print(f"Taxa de erro (δ): {error_rate:.6f}")

    return error_rate

print("\n--- Teste de Taxa de Erro ---")
test_error_rate(num_tests=500, lambda_param=256, message_len=256)


--- Teste de Taxa de Erro ---
Testes realizados: 500
Erros: 0
Taxa de erro (δ): 0.000000


0.0

### Quantum error correction